# GD Arena — Train Your Own ML Model
**A Python / Google Colab project for interview-style Group Discussion practice.**

This notebook trains a real supervised stance classifier. Give it an **English argument + topic target**;
it predicts **support, oppose or neutral**. No paid AI API, GPU or pretrained LLM is required.

The model stage comes first. Voice, Hinglish translation, virtual participants and the final UI
will be separate components. This model does not generate replies or assess employability.

**Use:** Upload this `.ipynb` to Google Colab, use a Python 3 CPU runtime, then choose **Runtime → Run all**.
The notebook writes all required source/data files itself. Internet is required to install packages
and download the original research dataset. It saves a downloadable ZIP at the end.

### What you can show in your presentation
1. Topic-conditioned classification, rather than positive/negative sentiment.
2. Dataset provenance, annotation labels, cleaning and split checks.
3. Training-only TF-IDF features and two learned classifiers.
4. Validation-based selection, followed by held-out precision, recall, F1 and confusion matrices.
5. Saved-model inference on a new argument.


## 1. Set up Python dependencies
The versions below match the delivered model. If Colab requests a session restart, restart and run all again.


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

requirements = ['numpy==2.5.4', 'scipy==1.16.3', 'scikit-learn==1.7.2',
                'joblib==1.6.0', 'matplotlib==3.10.7']
missing = []
for requirement in requirements:
    name, version = requirement.split('==')
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready. Python:', sys.version.split()[0])


## 2. Create the project files
The following cell writes the complete Python module. The module keeps training and inference
identical and allows a saved model to be loaded from another Python program later.


In [ ]:
from pathlib import Path
import os
import json

PROJECT = Path.cwd() / 'gd_arena'
(PROJECT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT / 'artifacts').mkdir(exist_ok=True)
print('Project folder:', PROJECT)


In [ ]:
%%writefile gd_arena/gd_model.py
"""GD Arena: reproducible, CPU-based, topic-conditioned stance classification.

Run: python gd_model.py train --data-dir data/vast --output-dir artifacts
No LLM, paid API, GPU or pretrained language model is used.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import html
import json
import re
import time
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import joblib
import numpy as np
import scipy
import sklearn
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

LABELS = ['oppose', 'support', 'neutral']
LABEL_MAP = {'0': 'oppose', '1': 'support', '2': 'neutral'}
SEED = 42
BASE = Path(__file__).resolve().parent
REVISION = 'e7c4775182b184730f350995f8260579c9e066fe'
CHECKSUMS = {
    'vast_train.csv': 'b2f0d69a73231b8ef22f10e4df7d759cf228505ce5252506068da6aa7fe42109',
    'vast_dev.csv': 'ea6f4c4aa7d35c1f961a7e041099b7e8a7420cdb1199b96b8f4f1d0e7d01d02c',
    'vast_test.csv': '32931d3390c5bf71fcfbb98e517ae38a11388d4258afbb5b0d612a6211335cec',
}


def clean(text):
    """Normalize formatting, retaining negation, punctuation and sentence meaning."""
    return re.sub(r'\s+', ' ', html.unescape(str(text))).strip()


def key(text):
    return re.sub(r'[^a-z0-9]+', ' ', clean(text).lower()).strip()


def download_vast(data_dir):
    """Download the authors' pinned release; validate every cached/downloaded file."""
    folder = Path(data_dir)
    folder.mkdir(parents=True, exist_ok=True)
    for filename, expected in CHECKSUMS.items():
        path = folder / filename
        if not path.exists():
            url = f'https://raw.githubusercontent.com/emilyallaway/zero-shot-stance/{REVISION}/data/VAST/{filename}'
            request = Request(url, headers={'User-Agent': 'GD-Arena-Educational-Project'})
            with urlopen(request, timeout=120) as response:
                payload = response.read()
            if hashlib.sha256(payload).hexdigest() != expected:
                raise ValueError(f'Checksum mismatch for downloaded {filename}')
            temp = path.with_suffix('.tmp')
            temp.write_bytes(payload)
            temp.replace(path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f'{filename} has changed. Use the pinned original data.')
    return folder


def read_vast(data_dir):
    """Keep original posts; upstream preprocessed text removes useful negations."""
    splits, audit = {}, {}
    previous_documents = set()
    for split in ['train', 'dev', 'test']:
        with (Path(data_dir) / f'vast_{split}.csv').open(encoding='utf-8', newline='') as handle:
            raw = list(csv.DictReader(handle))
        records, rejected, labels_by_pair = [], Counter(), {}
        for index, row in enumerate(raw):
            # VAST neutral examples may retain a stale `new_topic` field.
            # The authors' loader uses the JSON `topic` tokens as the true target.
            # Using new_topic would pair neutral labels with the wrong target.
            target_tokens = json.loads(row.get('topic', '[]'))
            if not isinstance(target_tokens, list) or not all(isinstance(t, str) for t in target_tokens):
                raise ValueError('Invalid VAST target token list')
            text, topic = clean(row.get('post', '')), clean(' '.join(target_tokens))
            label = LABEL_MAP.get(row.get('label', ''))
            if not text or not topic or label is None:
                rejected['invalid'] += 1
                continue
            pair = (key(text), key(topic))
            labels_by_pair.setdefault(pair, set()).add(label)
            records.append({'id': f'vast-{split}-{index}', 'text': text, 'topic': topic,
                            'label': label, 'source': 'VAST', 'group': key(text),
                            'official_seen': row.get('seen?', '')})
        used_pairs, accepted = set(), []
        for row in records:
            pair = (key(row['text']), key(row['topic']))
            if len(labels_by_pair[pair]) > 1:
                rejected['conflicting_pair'] += 1
            elif pair in used_pairs:
                rejected['duplicate_pair'] += 1
            elif row['group'] in previous_documents:
                rejected['document_in_earlier_split'] += 1
            else:
                accepted.append(row)
                used_pairs.add(pair)
        previous_documents.update(r['group'] for r in accepted)
        splits[split] = accepted
        audit[split] = {'original_rows': len(raw), 'used_rows': len(accepted),
                        'removed': dict(rejected),
                        'class_counts': dict(Counter(r['label'] for r in accepted)),
                        'unique_targets': len({key(r['topic']) for r in accepted}),
                        'unique_documents': len({r['group'] for r in accepted})}
    return splits, audit


def read_gd(path):
    """Read explicitly authored examples with immutable argument-family splits."""
    with Path(path).open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    splits = {s: [] for s in ['train', 'dev', 'test']}
    groups, documents = {}, {}
    for row in rows:
        split = row['split']
        if split not in splits or row['label'] not in LABELS:
            raise ValueError('Invalid split/label in GD data')
        if not row['text'].strip() or not row['topic'].strip():
            raise ValueError('Empty GD text or topic')
        group = row['group_id']
        doc = key(row['text'])
        if group in groups and groups[group] != split:
            raise ValueError('Argument family leaked across GD splits')
        if doc in documents and documents[doc] != split:
            raise ValueError('Duplicate GD text leaked across splits')
        groups[group], documents[doc] = split, split
        splits[split].append({**row, 'group': group, 'source': 'assistant_authored_synthetic'})
    return splits


def pair_terms(row):
    """Topic x argument lexical interactions; no labels or handcrafted stance rules."""
    topics = sorted(set(re.findall(r'[a-z]{3,}', row['topic'].lower())) - ENGLISH_STOP_WORDS)
    words = list(dict.fromkeys(re.findall(r'[a-z]{3,}', row['text'].lower())))[:100]
    return ' '.join(f'{t}__{w}' for t in topics[:12] for w in words if w not in ENGLISH_STOP_WORDS)


class StanceFeatures(BaseEstimator, TransformerMixin):
    """Fit all vocabularies/IDF on training examples only."""
    def fit(self, rows, y=None):
        self.arguments_ = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000,
                                         sublinear_tf=True, strip_accents='unicode', dtype=np.float32)
        self.targets_ = TfidfVectorizer(ngram_range=(1, 2), max_features=6000,
                                       sublinear_tf=True, strip_accents='unicode', dtype=np.float32)
        self.characters_ = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                           min_df=3, max_features=20000, sublinear_tf=True,
                                           dtype=np.float32)
        self.pairs_ = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b', min_df=3,
                                     max_features=40000, sublinear_tf=True, dtype=np.float32)
        self.arguments_.fit([r['text'] for r in rows])
        self.targets_.fit([r['topic'] for r in rows])
        self.characters_.fit([r['text'] for r in rows])
        self.pairs_.fit([pair_terms(r) for r in rows])
        return self

    def transform(self, rows):
        texts, topics = [r['text'] for r in rows], [r['topic'] for r in rows]
        matrix = hstack([self.arguments_.transform(texts),
                         self.targets_.transform(topics) * 0.5,
                         self.characters_.transform(texts) * 0.5,
                         self.pairs_.transform([pair_terms(r) for r in rows])], format='csr')
        return normalize(matrix, copy=False)


def evaluate(gold, predicted):
    return {'n': len(gold), 'accuracy': float(accuracy_score(gold, predicted)),
            'macro_f1': float(f1_score(gold, predicted, labels=LABELS, average='macro', zero_division=0)),
            'per_class': classification_report(gold, predicted, labels=LABELS, output_dict=True, zero_division=0),
            'confusion_matrix': confusion_matrix(gold, predicted, labels=LABELS).tolist(),
            'confusion_order': LABELS}


def make_plots(report, output):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': '#f8fafc',
                         'axes.facecolor': '#ffffff', 'axes.spines.top': False, 'axes.spines.right': False})
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), constrained_layout=True)
    for ax, dataset, title in zip(axes, ['vast', 'gd'], ['Public VAST test set', 'Authored GD test examples']):
        metrics = report['test'][dataset]
        cm = np.array(metrics['confusion_matrix'])
        ax.imshow(cm, cmap='Blues')
        for i in range(3):
            for j in range(3):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > cm.max() / 2 else '#16324f', fontsize=13)
        ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
               xlabel='Predicted stance', ylabel='Actual stance',
               title=f'{title}\nn={metrics["n"]:,} | macro F1={metrics["macro_f1"]:.3f}')
    fig.suptitle('GD Arena | Held-out evaluation', fontweight='bold', fontsize=16)
    fig.savefig(output / 'confusion_matrices.png', dpi=180)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    names = list(report['validation'])
    x = np.arange(len(names))
    ax.bar(x - .18, [report['validation'][n]['vast']['macro_f1'] for n in names], .36,
           color='#245dd3', label='Public VAST dev')
    ax.bar(x + .18, [report['validation'][n]['gd']['macro_f1'] for n in names], .36,
           color='#19a99c', label='Authored GD dev')
    ax.set(xticks=x, xticklabels=names, ylim=(0, 1), ylabel='Macro F1',
           title='Model selection uses validation data only')
    ax.legend()
    fig.savefig(output / 'model_comparison.png', dpi=180)
    plt.close(fig)


def train(data_dir, gd_path, output_dir):
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    vast, audit = read_vast(download_vast(data_dir))
    gd = read_gd(gd_path)
    training = vast['train'] + gd['train']
    for left, right in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:
        assert not ({key(r['text']) for r in vast[left] + gd[left]} &
                    {key(r['text']) for r in vast[right] + gd[right]}), 'Cross-split document leakage'
    print('Fitting TF-IDF features on training data only...', flush=True)
    started = time.monotonic()
    features = StanceFeatures().fit(training)
    x_train = features.transform(training)
    y_train = np.array([r['label'] for r in training])
    dev_x = {name: features.transform(rows['dev']) for name, rows in [('vast', vast), ('gd', gd)]}
    dev_y = {name: [r['label'] for r in rows['dev']] for name, rows in [('vast', vast), ('gd', gd)]}
    # Fixed BEFORE evaluation: modest emphasis on the small domain-adaptation set.
    sample_weights = np.array([1.0] * len(vast['train']) + [8.0] * len(gd['train']))
    with sklearn.config_context(enable_metadata_routing=True):
        logistic = OneVsRestClassifier(LogisticRegression(
            C=4.0, solver='liblinear', max_iter=1000, tol=0.002,
            random_state=SEED).set_fit_request(sample_weight=True))
    candidates = {
        'Majority baseline': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': logistic,
        'Linear SVM': LinearSVC(C=0.5, max_iter=5000, random_state=SEED, dual='auto'),
    }
    validation, fitted, training_warnings = {}, {}, {}
    for name, classifier in candidates.items():
        print(f'Training {name}...', flush=True)
        with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=2), sklearn.config_context(enable_metadata_routing=True):
            warnings.simplefilter('always')
            classifier.fit(x_train, y_train, sample_weight=sample_weights)
        training_warnings[name] = [str(w.message) for w in caught]
        fitted[name] = classifier
        validation[name] = {d: evaluate(dev_y[d], classifier.predict(dev_x[d])) for d in dev_x}
        validation[name]['selection_score'] = (validation[name]['vast']['macro_f1'] + validation[name]['gd']['macro_f1']) / 2
        print(name, 'VAST dev F1:', round(validation[name]['vast']['macro_f1'], 4),
              'GD dev F1:', round(validation[name]['gd']['macro_f1'], 4), flush=True)
    learned = ['Logistic Regression', 'Linear SVM']
    best_name = max(learned, key=lambda name: validation[name]['selection_score'])
    classifier = fitted[best_name]
    print(f'Selected on dev data: {best_name}. Evaluating test sets once.', flush=True)
    tests, test_predictions = {}, {}
    for name, rows in [('vast', vast['test']), ('gd', gd['test'])]:
        predicted = classifier.predict(features.transform(rows))
        test_predictions[name] = predicted
        tests[name] = evaluate([r['label'] for r in rows], predicted)
        # Export IDs and labels, not the third-party comment text.
        with (output / f'{name}_test_predictions.csv').open('w', newline='', encoding='utf-8') as handle:
            writer = csv.writer(handle)
            writer.writerow(['id', 'actual', 'predicted', 'correct'])
            writer.writerows((r['id'], r['label'], str(p), r['label'] == p) for r, p in zip(rows, predicted))
    train_targets = {key(r['topic']) for r in training}
    unseen = [i for i, r in enumerate(vast['test']) if key(r['topic']) not in train_targets]
    if unseen:
        tests['vast_unseen_target_strings'] = evaluate([vast['test'][i]['label'] for i in unseen],
                                                        [test_predictions['vast'][i] for i in unseen])
    tests['gd_per_topic'] = {}
    for topic_id in sorted({r['topic_id'] for r in gd['test']}):
        indices = [i for i, r in enumerate(gd['test']) if r['topic_id'] == topic_id]
        tests['gd_per_topic'][topic_id] = evaluate([gd['test'][i]['label'] for i in indices],
                                                   [test_predictions['gd'][i] for i in indices])
    report = {
        'created_utc': datetime.now(timezone.utc).isoformat(), 'seed': SEED,
        'selected_model': best_name, 'training_seconds': round(time.monotonic() - started, 2),
        'dataset_revision': REVISION, 'data_audit': audit,
        'dataset_fields': {'argument': 'post', 'target': 'JSON topic tokens joined by spaces',
                           'label': 'label; 0=oppose, 1=support, 2=neutral'},
        'gd_counts': {s: len(gd[s]) for s in gd},
        'gd_sha256': hashlib.sha256(Path(gd_path).read_bytes()).hexdigest(),
        'feature_count': int(x_train.shape[1]), 'training_rows': len(training),
        'selection_rule': 'Mean of VAST dev macro F1 and authored GD dev macro F1; test data never used for selection.',
        'source_weights': {'VAST': 1.0, 'assistant_authored_synthetic': 8.0},
        'validation': validation, 'test': tests, 'warnings': training_warnings,
        'versions': {'python': __import__('platform').python_version(), 'numpy': np.__version__,
                     'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__},
        'limitations': [
            'Experimental educational baseline; predictions may be wrong, including on simple arguments.',
            'GD examples and labels were authored by the assistant, not collected from real users or independently annotated.',
            'Authored GD test scores are a small development check, not evidence of real-world interview performance.',
            'English only. Hindi/Hinglish, voice recognition, translation and conversational turn generation are not trained here.',
            'Neutral means no clear stance towards the target; it does not mean low confidence.',
            'Raw decision scores are uncalibrated model scores, not probabilities or argument-quality scores.',
            'Bag-of-words features cannot reliably understand sarcasm, implication, long context or arbitrary reversed propositions.',
            'Unseen-target reporting uses exact normalized target strings, not semantic topic equivalence.',
            'VAST cleaning modifies the published evaluation set; metrics are not directly comparable with paper results.'
        ]}
    (output / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    bundle = {'features': features, 'classifier': classifier, 'model_name': best_name,
              'report': report, 'topics': json.loads((BASE / 'topics.json').read_text(encoding='utf-8'))}
    joblib.dump(bundle, output / 'stance_model.joblib', compress=3)
    make_plots(report, output)
    print(json.dumps({n: {'accuracy': tests[n]['accuracy'], 'macro_f1': tests[n]['macro_f1'], 'n': tests[n]['n']}
                      for n in ['vast', 'gd']}, indent=2), flush=True)
    return bundle, report


def predict(bundle, text, topic_id=None, target=None):
    if not isinstance(text, str) or len(text.strip().split()) < 3:
        raise ValueError('Enter an English argument of at least three words.')
    if len(text) > 5000:
        raise ValueError('Use at most 5,000 characters per argument.')
    if topic_id is not None:
        match = next((t for t in bundle['topics'] if t['id'] == topic_id), None)
        if match is None:
            raise ValueError(f'Unknown topic ID: {topic_id}')
        target = match['target']
    if not isinstance(target, str) or not target.strip():
        raise ValueError('Choose a topic_id or supply an explicit target proposition.')
    if len(target) > 500:
        raise ValueError('Target must be 500 characters or fewer.')
    x = bundle['features'].transform([{'text': clean(text), 'topic': clean(target)}])
    classifier = bundle['classifier']
    scores = classifier.decision_function(x)[0]
    return {'target': target, 'stance': str(classifier.predict(x)[0]),
            'decision_scores': {str(k): round(float(v), 4) for k, v in zip(classifier.classes_, scores)},
            'score_note': 'Uncalibrated scores; not confidence percentages.',
            'language': 'English only', 'model': bundle['model_name']}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest='command', required=True)
    training = commands.add_parser('train')
    training.add_argument('--data-dir', default=str(BASE / 'data' / 'vast'))
    training.add_argument('--gd-data', default=str(BASE / 'data' / 'gd_examples.csv'))
    training.add_argument('--output-dir', default=str(BASE / 'artifacts'))
    inference = commands.add_parser('predict')
    inference.add_argument('--model', default=str(BASE / 'artifacts' / 'stance_model.joblib'))
    inference.add_argument('--topic', required=True)
    inference.add_argument('--text', required=True)
    commands.add_parser('topics')
    args = parser.parse_args()
    if args.command == 'train':
        train(args.data_dir, args.gd_data, args.output_dir)
    elif args.command == 'predict':
        bundle = joblib.load(args.model)
        print(json.dumps(predict(bundle, args.text, topic_id=args.topic), indent=2))
    else:
        print((BASE / 'topics.json').read_text(encoding='utf-8'))


if __name__ == '__main__':
    # Keep saved custom classes importable when training through the CLI.
    from gd_model import main as imported_main
    imported_main()


## 3. Define the ten GD topics
Each title has an explicit proposition. For example, support for the remote-work topic means
support for **working from home being better than office work**. That removes ambiguity from labels.


In [ ]:
TOPICS = [{'id': 'ai_jobs', 'title': 'AI: Job Creator or Job Destroyer?', 'target': 'Artificial intelligence will create more jobs than it destroys.', 'intro': 'Consider automation, new occupations, reskilling and who benefits.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'remote_work', 'title': 'Work From Home vs Work From Office', 'target': 'Working from home is better than working from an office.', 'intro': 'Compare productivity, collaboration, inclusion and work-life boundaries.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'online_education', 'title': 'Online Education vs Classroom Learning', 'target': 'Online education is better than classroom learning.', 'intro': 'Discuss access, practical learning, engagement and assessment.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'social_media', 'title': 'Social Media: Connecting Us or Isolating Us?', 'target': 'Social media strengthens meaningful human connections.', 'intro': 'Consider community, attention, relationships and loneliness.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'skills_degrees', 'title': 'Skills vs Degrees: What Matters More?', 'target': 'Employers should prioritize practical skills over academic degrees.', 'intro': 'Discuss evidence of ability, credentials, fairness and training.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'ai_education', 'title': 'Should AI Tools Be Allowed in Education?', 'target': 'Students should be allowed to use AI tools in education.', 'intro': 'Consider tutoring, original thinking, assessment and disclosure.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'digital_payments', 'title': 'Digital Payments vs Cash', 'target': 'Digital payments are better than cash for everyday transactions.', 'intro': 'Discuss convenience, accessibility, privacy and reliability.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'startups', 'title': 'Startups vs Corporate Jobs', 'target': 'Joining a startup is a better career choice than joining a large corporation.', 'intro': 'Consider learning, stability, responsibility and career goals.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'ev_india', 'title': 'Electric Vehicles: Is India Ready?', 'target': 'India is ready for widespread adoption of electric vehicles.', 'intro': 'Discuss charging, affordability, electricity supply and infrastructure.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}, {'id': 'environment', 'title': 'Economic Growth vs Environmental Protection', 'target': 'Environmental protection should take priority over short-term economic growth.', 'intro': 'Consider jobs, health, natural resources and long-term costs.', 'label_meaning': {'support': 'Supports the target proposition.', 'oppose': 'Opposes the target proposition.', 'neutral': 'Does not express a clear stance towards the target.'}}]
(PROJECT / 'topics.json').write_text(json.dumps(TOPICS, indent=2), encoding='utf-8')
for topic in TOPICS:
    print(f"{topic['id']:20s} {topic['title']}")


## 4. Add the starter GD data
**These 300 examples and labels were authored by the assistant. They are synthetic educational
material, not independently annotated real-user responses.** They supplement the public VAST corpus.
There are 180 training, 60 validation and 60 test examples; each split contains all ten topics.
Keep related/paraphrased examples in one split. For a stronger project, independently label real GD
responses and reserve a fresh final test set.


In [ ]:
GD_CSV = r'''id,topic_id,topic,text,label,split,group_id,provenance
gd-ai_jobs-support-00,ai_jobs,Artificial intelligence will create more jobs than it destroys.,AI will create new careers in model auditing that did not exist a decade ago.,support,train,ai_jobs-family-00,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-01,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Small businesses can expand their services with AI and hire more people to serve new customers.,support,train,ai_jobs-family-01,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-02,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Workers who automate repetitive tasks can spend more time designing products and building new businesses.,support,train,ai_jobs-family-02,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-03,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Demand for people who maintain and supervise intelligent systems can outweigh the jobs automated away.,support,train,ai_jobs-family-03,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-04,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Affordable AI tools let entrepreneurs launch companies that would otherwise never employ anyone.,support,train,ai_jobs-family-04,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-05,ai_jobs,Artificial intelligence will create more jobs than it destroys.,"As AI makes services cheaper, more customers can afford them and businesses need additional workers.",support,train,ai_jobs-family-05,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-06,ai_jobs,Artificial intelligence will create more jobs than it destroys.,"Automation in earlier industries created complementary occupations, and AI can repeat that pattern.",support,dev,ai_jobs-family-06,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-07,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Training displaced staff for new AI-assisted roles can produce a net increase in employment.,support,dev,ai_jobs-family-07,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-08,ai_jobs,Artificial intelligence will create more jobs than it destroys.,New markets for personalized services will require human specialists even when machines handle routine work.,support,test,ai_jobs-family-08,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-support-09,ai_jobs,Artificial intelligence will create more jobs than it destroys.,The expansion of AI safety and integration teams can generate more positions than routine automation removes.,support,test,ai_jobs-family-09,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-00,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Companies can replace entire support teams with AI without creating equally large replacement teams.,oppose,train,ai_jobs-family-00,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-01,ai_jobs,Artificial intelligence will create more jobs than it destroys.,New AI engineering jobs require skills that many displaced workers cannot quickly acquire.,oppose,train,ai_jobs-family-01,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-02,ai_jobs,Artificial intelligence will create more jobs than it destroys.,One automated system can perform the routine tasks of hundreds of entry-level employees.,oppose,train,ai_jobs-family-02,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-03,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Firms have a financial incentive to cut payroll when AI becomes cheaper than human labor.,oppose,train,ai_jobs-family-03,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-04,ai_jobs,Artificial intelligence will create more jobs than it destroys.,The number of new specialist roles may be much smaller than the number of clerical jobs lost.,oppose,train,ai_jobs-family-04,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-05,ai_jobs,Artificial intelligence will create more jobs than it destroys.,"Automated content production can reduce opportunities for translators, writers and designers at the same time.",oppose,train,ai_jobs-family-05,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-06,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Reskilling takes years while companies can deploy labor-saving AI systems within months.,oppose,dev,ai_jobs-family-06,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-07,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Productivity gains may increase profits for owners without creating additional jobs.,oppose,dev,ai_jobs-family-07,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-08,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Graduates may lose the junior roles through which they would normally build experience.,oppose,test,ai_jobs-family-08,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-oppose-09,ai_jobs,Artificial intelligence will create more jobs than it destroys.,A few people can oversee automated operations that previously required a large workforce.,oppose,test,ai_jobs-family-09,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-00,ai_jobs,Artificial intelligence will create more jobs than it destroys.,We should define whether a job means a full-time position or a short freelance assignment.,neutral,train,ai_jobs-family-00,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-01,ai_jobs,Artificial intelligence will create more jobs than it destroys.,"AI employment effects can be studied separately in manufacturing, finance and education.",neutral,train,ai_jobs-family-01,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-02,ai_jobs,Artificial intelligence will create more jobs than it destroys.,The time period matters when comparing jobs created with jobs displaced.,neutral,train,ai_jobs-family-02,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-03,ai_jobs,Artificial intelligence will create more jobs than it destroys.,This discussion concerns employment numbers rather than whether AI is conscious.,neutral,train,ai_jobs-family-03,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-04,ai_jobs,Artificial intelligence will create more jobs than it destroys.,We need a consistent method for counting new occupations and changed job descriptions.,neutral,train,ai_jobs-family-04,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-05,ai_jobs,Artificial intelligence will create more jobs than it destroys.,I would like to know which country and which industries this comparison covers.,neutral,train,ai_jobs-family-05,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-06,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Some jobs will change their tasks even if their job titles remain the same.,neutral,dev,ai_jobs-family-06,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-07,ai_jobs,Artificial intelligence will create more jobs than it destroys.,Can the group distinguish direct AI jobs from work indirectly supported by AI?,neutral,dev,ai_jobs-family-07,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-08,ai_jobs,Artificial intelligence will create more jobs than it destroys.,A forecast would need assumptions about adoption speed and population growth.,neutral,test,ai_jobs-family-08,assistant_authored_synthetic; requires independent human review
gd-ai_jobs-neutral-09,ai_jobs,Artificial intelligence will create more jobs than it destroys.,"Before drawing a conclusion, let us clarify what counts as an AI-related job loss.",neutral,test,ai_jobs-family-09,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-00,remote_work,Working from home is better than working from an office.,Working from home saves commuting time that employees can use for rest and focused work.,support,train,remote_work-family-00,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-01,remote_work,Working from home is better than working from an office.,Remote workers can choose a quiet setting instead of constant interruptions in an open office.,support,train,remote_work-family-01,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-02,remote_work,Working from home is better than working from an office.,Home working makes jobs accessible to people who cannot relocate to an expensive city.,support,train,remote_work-family-02,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-03,remote_work,Working from home is better than working from an office.,Flexible remote schedules help employees balance family responsibilities with professional commitments.,support,train,remote_work-family-03,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-04,remote_work,Working from home is better than working from an office.,Companies can hire talented people across regions when daily office attendance is unnecessary.,support,train,remote_work-family-04,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-05,remote_work,Working from home is better than working from an office.,Avoiding daily travel reduces stress and makes remote work a more sustainable routine.,support,train,remote_work-family-05,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-06,remote_work,Working from home is better than working from an office.,Employees can arrange their home workspace to suit accessibility needs that a shared office may overlook.,support,dev,remote_work-family-06,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-07,remote_work,Working from home is better than working from an office.,Remote work allows deeper concentration when teams use written updates instead of frequent interruptions.,support,dev,remote_work-family-07,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-08,remote_work,Working from home is better than working from an office.,Parents can stay in the workforce more easily when work location is flexible.,support,test,remote_work-family-08,assistant_authored_synthetic; requires independent human review
gd-remote_work-support-09,remote_work,Working from home is better than working from an office.,The money saved on daily travel and meals makes working at home more attractive for many employees.,support,test,remote_work-family-09,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-00,remote_work,Working from home is better than working from an office.,New employees learn faster when they can ask experienced colleagues questions in person.,oppose,train,remote_work-family-00,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-01,remote_work,Working from home is better than working from an office.,Working from home can blur boundaries and make people feel available after normal office hours.,oppose,train,remote_work-family-01,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-02,remote_work,Working from home is better than working from an office.,Some employees do not have a quiet room or reliable internet at home.,oppose,train,remote_work-family-02,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-03,remote_work,Working from home is better than working from an office.,Face-to-face discussions can resolve complicated teamwork problems faster than remote message threads.,oppose,train,remote_work-family-03,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-04,remote_work,Working from home is better than working from an office.,Office attendance helps build trust between colleagues who have never worked together before.,oppose,train,remote_work-family-04,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-05,remote_work,Working from home is better than working from an office.,Loneliness at home can reduce motivation and make a shared office the healthier option.,oppose,train,remote_work-family-05,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-06,remote_work,Working from home is better than working from an office.,Hardware teams need physical equipment that cannot be sent to every employee's home.,oppose,dev,remote_work-family-06,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-07,remote_work,Working from home is better than working from an office.,An office gives employees a clear separation between their job and domestic responsibilities.,oppose,dev,remote_work-family-07,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-08,remote_work,Working from home is better than working from an office.,Informal workplace conversations can reveal problems that scheduled video meetings miss.,oppose,test,remote_work-family-08,assistant_authored_synthetic; requires independent human review
gd-remote_work-oppose-09,remote_work,Working from home is better than working from an office.,Security-sensitive work may be easier to supervise in a controlled company office.,oppose,test,remote_work-family-09,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-00,remote_work,Working from home is better than working from an office.,We should specify whether this comparison is about fully remote work or a hybrid arrangement.,neutral,train,remote_work-family-00,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-01,remote_work,Working from home is better than working from an office.,Different job roles have different requirements for equipment and direct contact with customers.,neutral,train,remote_work-family-01,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-02,remote_work,Working from home is better than working from an office.,The team needs an agreed definition of productivity before comparing work locations.,neutral,train,remote_work-family-02,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-03,remote_work,Working from home is better than working from an office.,How many days per week does the proposed office policy require?,neutral,train,remote_work-family-03,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-04,remote_work,Working from home is better than working from an office.,Employee preferences can be collected through a survey before this discussion.,neutral,train,remote_work-family-04,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-05,remote_work,Working from home is better than working from an office.,The phrase work from home refers to location rather than the number of working hours.,neutral,train,remote_work-family-05,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-06,remote_work,Working from home is better than working from an office.,Let us separate an employee's commute from the tasks performed during the working day.,neutral,dev,remote_work-family-06,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-07,remote_work,Working from home is better than working from an office.,We could compare two departments doing similar work under different location policies.,neutral,dev,remote_work-family-07,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-08,remote_work,Working from home is better than working from an office.,"Are we discussing experienced employees, new graduates or both groups?",neutral,test,remote_work-family-08,assistant_authored_synthetic; requires independent human review
gd-remote_work-neutral-09,remote_work,Working from home is better than working from an office.,The discussion should state whether employers provide equipment for home offices.,neutral,test,remote_work-family-09,assistant_authored_synthetic; requires independent human review
gd-online_education-support-00,online_education,Online education is better than classroom learning.,Recorded online lessons let students revisit difficult explanations at their own pace.,support,train,online_education-family-00,assistant_authored_synthetic; requires independent human review
gd-online_education-support-01,online_education,Online education is better than classroom learning.,Online courses bring specialist teachers to learners who live far from major schools.,support,train,online_education-family-01,assistant_authored_synthetic; requires independent human review
gd-online_education-support-02,online_education,Online education is better than classroom learning.,Digital classes remove travel barriers for students who have mobility limitations.,support,train,online_education-family-02,assistant_authored_synthetic; requires independent human review
gd-online_education-support-03,online_education,Online education is better than classroom learning.,Learners with jobs can study online without attending a classroom at fixed daytime hours.,support,train,online_education-family-03,assistant_authored_synthetic; requires independent human review
gd-online_education-support-04,online_education,Online education is better than classroom learning.,Online platforms can provide instant practice feedback and adapt exercises to each learner.,support,train,online_education-family-04,assistant_authored_synthetic; requires independent human review
gd-online_education-support-05,online_education,Online education is better than classroom learning.,Lower accommodation and commuting costs can make online study affordable for more families.,support,train,online_education-family-05,assistant_authored_synthetic; requires independent human review
gd-online_education-support-06,online_education,Online education is better than classroom learning.,A searchable lesson archive helps students recover material they missed during illness.,support,dev,online_education-family-06,assistant_authored_synthetic; requires independent human review
gd-online_education-support-07,online_education,Online education is better than classroom learning.,Online discussion boards give quieter students time to prepare thoughtful contributions.,support,dev,online_education-family-07,assistant_authored_synthetic; requires independent human review
gd-online_education-support-08,online_education,Online education is better than classroom learning.,Remote courses allow learners to choose subjects unavailable at their local institution.,support,test,online_education-family-08,assistant_authored_synthetic; requires independent human review
gd-online_education-support-09,online_education,Online education is better than classroom learning.,Interactive online exercises can give each student more practice than a crowded classroom allows.,support,test,online_education-family-09,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-00,online_education,Online education is better than classroom learning.,Students without reliable internet cannot participate equally in online lessons.,oppose,train,online_education-family-00,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-01,online_education,Online education is better than classroom learning.,Classrooms make it easier for teachers to notice confusion and respond immediately.,oppose,train,online_education-family-01,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-02,online_education,Online education is better than classroom learning.,Practical laboratory skills require supervised experience with real equipment.,oppose,train,online_education-family-02,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-03,online_education,Online education is better than classroom learning.,Home distractions can make online learning harder than studying in a dedicated classroom.,oppose,train,online_education-family-03,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-04,online_education,Online education is better than classroom learning.,Young students benefit from the social interaction and routines of a physical school.,oppose,train,online_education-family-04,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-05,online_education,Online education is better than classroom learning.,Online assessment can make it difficult to verify that students completed their own work.,oppose,train,online_education-family-05,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-06,online_education,Online education is better than classroom learning.,Not every learner has the self-discipline needed to follow recorded lessons alone.,oppose,dev,online_education-family-06,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-07,online_education,Online education is better than classroom learning.,Teachers can provide more immediate hands-on support during an in-person workshop.,oppose,dev,online_education-family-07,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-08,online_education,Online education is better than classroom learning.,Too much screen time can reduce attention during long online classes.,oppose,test,online_education-family-08,assistant_authored_synthetic; requires independent human review
gd-online_education-oppose-09,online_education,Online education is better than classroom learning.,Students may hesitate to ask questions when they feel disconnected from a remote teacher.,oppose,test,online_education-family-09,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-00,online_education,Online education is better than classroom learning.,Online education includes both recorded courses and live video classes.,neutral,train,online_education-family-00,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-01,online_education,Online education is better than classroom learning.,We need to specify the age group before comparing learning outcomes.,neutral,train,online_education-family-01,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-02,online_education,Online education is better than classroom learning.,"The discussion could compare examination results, attendance or student satisfaction.",neutral,train,online_education-family-02,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-03,online_education,Online education is better than classroom learning.,How is practical work included in the proposed online course?,neutral,train,online_education-family-03,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-04,online_education,Online education is better than classroom learning.,Some institutions combine digital material with scheduled classroom meetings.,neutral,train,online_education-family-04,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-05,online_education,Online education is better than classroom learning.,Course length and subject difficulty should be similar in any comparison.,neutral,train,online_education-family-05,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-06,online_education,Online education is better than classroom learning.,Let us clarify whether the online option includes access to a teacher.,neutral,dev,online_education-family-06,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-07,online_education,Online education is better than classroom learning.,Assessment methods differ across universities and training providers.,neutral,dev,online_education-family-07,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-08,online_education,Online education is better than classroom learning.,"The word education covers school, university and professional training.",neutral,test,online_education-family-08,assistant_authored_synthetic; requires independent human review
gd-online_education-neutral-09,online_education,Online education is better than classroom learning.,We should establish whether this comparison concerns cost or learning quality.,neutral,test,online_education-family-09,assistant_authored_synthetic; requires independent human review
gd-social_media-support-00,social_media,Social media strengthens meaningful human connections.,Social media helps friends living in different countries stay involved in each other's lives.,support,train,social_media-family-00,assistant_authored_synthetic; requires independent human review
gd-social_media-support-01,social_media,Social media strengthens meaningful human connections.,People with rare interests can find supportive communities they would never meet locally.,support,train,social_media-family-01,assistant_authored_synthetic; requires independent human review
gd-social_media-support-02,social_media,Social media strengthens meaningful human connections.,Online groups help new residents form friendships before they feel comfortable in a city.,support,train,social_media-family-02,assistant_authored_synthetic; requires independent human review
gd-social_media-support-03,social_media,Social media strengthens meaningful human connections.,Sharing updates makes it easier to maintain relationships despite conflicting work schedules.,support,train,social_media-family-03,assistant_authored_synthetic; requires independent human review
gd-social_media-support-04,social_media,Social media strengthens meaningful human connections.,Social platforms let isolated individuals find peers who understand their experiences.,support,train,social_media-family-04,assistant_authored_synthetic; requires independent human review
gd-social_media-support-05,social_media,Social media strengthens meaningful human connections.,Community groups can turn online conversations into regular in-person meetings.,support,train,social_media-family-05,assistant_authored_synthetic; requires independent human review
gd-social_media-support-06,social_media,Social media strengthens meaningful human connections.,Family members separated by distance can celebrate important events together through social platforms.,support,dev,social_media-family-06,assistant_authored_synthetic; requires independent human review
gd-social_media-support-07,social_media,Social media strengthens meaningful human connections.,Support communities allow people to exchange encouragement when local support is limited.,support,dev,social_media-family-07,assistant_authored_synthetic; requires independent human review
gd-social_media-support-08,social_media,Social media strengthens meaningful human connections.,Messaging groups help old classmates preserve friendships long after leaving school.,support,test,social_media-family-08,assistant_authored_synthetic; requires independent human review
gd-social_media-support-09,social_media,Social media strengthens meaningful human connections.,Social networks make it easier to reconnect with people we have lost contact with.,support,test,social_media-family-09,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-00,social_media,Social media strengthens meaningful human connections.,Scrolling through feeds can replace the attentive face-to-face time that close relationships need.,oppose,train,social_media-family-00,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-01,social_media,Social media strengthens meaningful human connections.,Counting likes encourages superficial approval rather than meaningful friendship.,oppose,train,social_media-family-01,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-02,social_media,Social media strengthens meaningful human connections.,Curated posts can make people feel excluded even while they appear socially connected.,oppose,train,social_media-family-02,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-03,social_media,Social media strengthens meaningful human connections.,Online arguments can damage relationships that would be handled more calmly in person.,oppose,train,social_media-family-03,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-04,social_media,Social media strengthens meaningful human connections.,Constant notifications interrupt conversations with the people physically around us.,oppose,train,social_media-family-04,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-05,social_media,Social media strengthens meaningful human connections.,Having many online contacts does not prevent loneliness when few provide genuine support.,oppose,train,social_media-family-05,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-06,social_media,Social media strengthens meaningful human connections.,Recommendation feeds can trap users in isolated groups instead of exposing them to diverse relationships.,oppose,dev,social_media-family-06,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-07,social_media,Social media strengthens meaningful human connections.,Social comparison may increase insecurity and make users withdraw from real-world friendships.,oppose,dev,social_media-family-07,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-08,social_media,Social media strengthens meaningful human connections.,Public performance on social platforms can discourage honest conversations about personal difficulties.,oppose,test,social_media-family-08,assistant_authored_synthetic; requires independent human review
gd-social_media-oppose-09,social_media,Social media strengthens meaningful human connections.,Time spent seeking online attention can weaken the deeper relationships people already have.,oppose,test,social_media-family-09,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-00,social_media,Social media strengthens meaningful human connections.,"Social media includes messaging services, public feeds and community forums.",neutral,train,social_media-family-00,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-01,social_media,Social media strengthens meaningful human connections.,We should distinguish the number of contacts from the quality of relationships.,neutral,train,social_media-family-01,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-02,social_media,Social media strengthens meaningful human connections.,Which age group and which platform are we discussing?,neutral,train,social_media-family-02,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-03,social_media,Social media strengthens meaningful human connections.,Researchers can measure connection through surveys or observation of communication patterns.,neutral,train,social_media-family-03,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-04,social_media,Social media strengthens meaningful human connections.,Some people use social platforms for work and others mainly for family updates.,neutral,train,social_media-family-04,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-05,social_media,Social media strengthens meaningful human connections.,The group should agree on what meaningful human connection means.,neutral,train,social_media-family-05,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-06,social_media,Social media strengthens meaningful human connections.,Daily usage time alone does not describe what a person does on a platform.,neutral,dev,social_media-family-06,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-07,social_media,Social media strengthens meaningful human connections.,We could compare active conversations with passive reading of posts.,neutral,dev,social_media-family-07,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-08,social_media,Social media strengthens meaningful human connections.,The discussion concerns social relationships rather than platform revenue.,neutral,test,social_media-family-08,assistant_authored_synthetic; requires independent human review
gd-social_media-neutral-09,social_media,Social media strengthens meaningful human connections.,Let us clarify whether private messages and public comments belong in the same category.,neutral,test,social_media-family-09,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-00,skills_degrees,Employers should prioritize practical skills over academic degrees.,A practical work sample shows whether a candidate can perform the actual job.,support,train,skills_degrees-family-00,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-01,skills_degrees,Employers should prioritize practical skills over academic degrees.,Candidates without expensive degrees should have a chance to demonstrate their ability.,support,train,skills_degrees-family-01,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-02,skills_degrees,Employers should prioritize practical skills over academic degrees.,"Technology changes quickly, so current skills can matter more than an old qualification.",support,train,skills_degrees-family-02,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-03,skills_degrees,Employers should prioritize practical skills over academic degrees.,Hiring based on demonstrated competence can reduce unnecessary educational barriers.,support,train,skills_degrees-family-03,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-04,skills_degrees,Employers should prioritize practical skills over academic degrees.,A strong project portfolio reveals problem-solving ability that a certificate alone cannot show.,support,train,skills_degrees-family-04,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-05,skills_degrees,Employers should prioritize practical skills over academic degrees.,Experienced workers should not be rejected merely because they lack a university credential.,support,train,skills_degrees-family-05,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-06,skills_degrees,Employers should prioritize practical skills over academic degrees.,Apprentices can develop valuable abilities through real work without following an academic route.,support,dev,skills_degrees-family-06,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-07,skills_degrees,Employers should prioritize practical skills over academic degrees.,Testing relevant tasks gives employers direct evidence of a candidate's readiness.,support,dev,skills_degrees-family-07,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-08,skills_degrees,Employers should prioritize practical skills over academic degrees.,Self-taught programmers can solve business problems as effectively as graduates when their skills are strong.,support,test,skills_degrees-family-08,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-support-09,skills_degrees,Employers should prioritize practical skills over academic degrees.,Prioritizing practical ability broadens recruitment beyond people who could afford formal education.,support,test,skills_degrees-family-09,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-00,skills_degrees,Employers should prioritize practical skills over academic degrees.,A rigorous degree provides theoretical foundations that narrow task-based tests may miss.,oppose,train,skills_degrees-family-00,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-01,skills_degrees,Employers should prioritize practical skills over academic degrees.,Accredited qualifications help employers verify essential knowledge in highly specialized professions.,oppose,train,skills_degrees-family-01,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-02,skills_degrees,Employers should prioritize practical skills over academic degrees.,Years of structured study can develop research skills that a short practical assessment cannot reveal.,oppose,train,skills_degrees-family-02,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-03,skills_degrees,Employers should prioritize practical skills over academic degrees.,University education exposes students to concepts beyond the tools used in their current project.,oppose,train,skills_degrees-family-03,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-04,skills_degrees,Employers should prioritize practical skills over academic degrees.,Degrees provide a consistent signal when employers cannot thoroughly test every applicant.,oppose,train,skills_degrees-family-04,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-05,skills_degrees,Employers should prioritize practical skills over academic degrees.,Some professions require formal qualifications to demonstrate that required standards were studied.,oppose,train,skills_degrees-family-05,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-06,skills_degrees,Employers should prioritize practical skills over academic degrees.,Practical techniques may become obsolete while strong academic foundations remain useful.,oppose,dev,skills_degrees-family-06,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-07,skills_degrees,Employers should prioritize practical skills over academic degrees.,Completing a demanding degree can show persistence across a wide range of difficult subjects.,oppose,dev,skills_degrees-family-07,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-08,skills_degrees,Employers should prioritize practical skills over academic degrees.,Candidates may rehearse a portfolio project without understanding the principles behind it.,oppose,test,skills_degrees-family-08,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-oppose-09,skills_degrees,Employers should prioritize practical skills over academic degrees.,Formal education can teach careful reasoning that isolated demonstrations of skill overlook.,oppose,test,skills_degrees-family-09,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-00,skills_degrees,Employers should prioritize practical skills over academic degrees.,We should identify the profession before discussing its hiring requirements.,neutral,train,skills_degrees-family-00,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-01,skills_degrees,Employers should prioritize practical skills over academic degrees.,Degrees and practical skills can be measured in different ways.,neutral,train,skills_degrees-family-01,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-02,skills_degrees,Employers should prioritize practical skills over academic degrees.,What type of work sample would the hiring process use?,neutral,train,skills_degrees-family-02,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-03,skills_degrees,Employers should prioritize practical skills over academic degrees.,Some applicants have both a degree and substantial industry experience.,neutral,train,skills_degrees-family-03,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-04,skills_degrees,Employers should prioritize practical skills over academic degrees.,The discussion should distinguish entry-level hiring from recruitment of experienced staff.,neutral,train,skills_degrees-family-04,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-05,skills_degrees,Employers should prioritize practical skills over academic degrees.,An academic degree records completion of a particular course of study.,neutral,train,skills_degrees-family-05,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-06,skills_degrees,Employers should prioritize practical skills over academic degrees.,We need to define which skills are relevant to the advertised role.,neutral,dev,skills_degrees-family-06,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-07,skills_degrees,Employers should prioritize practical skills over academic degrees.,"Recruiters may use interviews, qualifications and work samples together.",neutral,dev,skills_degrees-family-07,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-08,skills_degrees,Employers should prioritize practical skills over academic degrees.,Are we discussing all degrees or only qualifications related to the job?,neutral,test,skills_degrees-family-08,assistant_authored_synthetic; requires independent human review
gd-skills_degrees-neutral-09,skills_degrees,Employers should prioritize practical skills over academic degrees.,Let us clarify how an employer would verify skills claimed on a resume.,neutral,test,skills_degrees-family-09,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-00,ai_education,Students should be allowed to use AI tools in education.,AI tutoring can explain a difficult concept in several ways until a student understands it.,support,train,ai_education-family-00,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-01,ai_education,Students should be allowed to use AI tools in education.,Students should learn to use AI responsibly because these tools will be part of many workplaces.,support,train,ai_education-family-01,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-02,ai_education,Students should be allowed to use AI tools in education.,AI can provide practice feedback when a teacher cannot respond to every student immediately.,support,train,ai_education-family-02,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-03,ai_education,Students should be allowed to use AI tools in education.,Learners with accessibility needs can use AI to adapt educational material to their requirements.,support,train,ai_education-family-03,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-04,ai_education,Students should be allowed to use AI tools in education.,Allowing AI with clear disclosure rules encourages honest use rather than hidden shortcuts.,support,train,ai_education-family-04,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-05,ai_education,Students should be allowed to use AI tools in education.,Teachers can use AI-assisted exercises to give students more personalized practice.,support,train,ai_education-family-05,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-06,ai_education,Students should be allowed to use AI tools in education.,Students can compare AI answers with reliable sources and develop critical evaluation skills.,support,dev,ai_education-family-06,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-07,ai_education,Students should be allowed to use AI tools in education.,AI language support can help learners understand course content written in an unfamiliar language.,support,dev,ai_education-family-07,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-08,ai_education,Students should be allowed to use AI tools in education.,Permission to use AI for brainstorming can help students begin a project without replacing their final reasoning.,support,test,ai_education-family-08,assistant_authored_synthetic; requires independent human review
gd-ai_education-support-09,ai_education,Students should be allowed to use AI tools in education.,Supervised AI use gives teachers opportunities to show students how to detect mistakes in generated answers.,support,test,ai_education-family-09,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-00,ai_education,Students should be allowed to use AI tools in education.,Students may submit AI-generated assignments without developing the skills the assignments were meant to teach.,oppose,train,ai_education-family-00,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-01,ai_education,Students should be allowed to use AI tools in education.,AI tools can provide convincing but incorrect explanations that inexperienced learners may trust.,oppose,train,ai_education-family-01,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-02,ai_education,Students should be allowed to use AI tools in education.,Allowing AI assistance can make it difficult to assess a student's independent understanding.,oppose,train,ai_education-family-02,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-03,ai_education,Students should be allowed to use AI tools in education.,Students with paid AI access may receive advantages unavailable to their classmates.,oppose,train,ai_education-family-03,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-04,ai_education,Students should be allowed to use AI tools in education.,Frequent AI use can make learners dependent on suggestions instead of solving problems themselves.,oppose,train,ai_education-family-04,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-05,ai_education,Students should be allowed to use AI tools in education.,Using AI to write answers can reduce the practice students need to develop their own reasoning.,oppose,train,ai_education-family-05,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-06,ai_education,Students should be allowed to use AI tools in education.,Schools may be unable to verify which parts of an assignment were produced by the student.,oppose,dev,ai_education-family-06,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-07,ai_education,Students should be allowed to use AI tools in education.,AI tutoring can miss a learner's misconception while presenting a fluent but unhelpful response.,oppose,dev,ai_education-family-07,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-08,ai_education,Students should be allowed to use AI tools in education.,Uploading classroom work to external AI services can expose student information unnecessarily.,oppose,test,ai_education-family-08,assistant_authored_synthetic; requires independent human review
gd-ai_education-oppose-09,ai_education,Students should be allowed to use AI tools in education.,Automatic essay generation can weaken writing development when it replaces planning and revision.,oppose,test,ai_education-family-09,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-00,ai_education,Students should be allowed to use AI tools in education.,"The term AI tool includes tutors, grammar assistants and systems that write entire assignments.",neutral,train,ai_education-family-00,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-01,ai_education,Students should be allowed to use AI tools in education.,We should specify whether the proposed policy covers homework or examinations.,neutral,train,ai_education-family-01,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-02,ai_education,Students should be allowed to use AI tools in education.,What disclosure would students need to provide when they use an AI assistant?,neutral,train,ai_education-family-02,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-03,ai_education,Students should be allowed to use AI tools in education.,Different courses assess different kinds of independent work.,neutral,train,ai_education-family-03,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-04,ai_education,Students should be allowed to use AI tools in education.,The institution needs a definition of permitted assistance before applying any policy.,neutral,train,ai_education-family-04,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-05,ai_education,Students should be allowed to use AI tools in education.,Some classroom activities already involve software that uses machine learning.,neutral,train,ai_education-family-05,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-06,ai_education,Students should be allowed to use AI tools in education.,Let us distinguish help understanding a concept from help producing a final answer.,neutral,dev,ai_education-family-06,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-07,ai_education,Students should be allowed to use AI tools in education.,We could discuss whether the same rules should apply to school pupils and university students.,neutral,dev,ai_education-family-07,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-08,ai_education,Students should be allowed to use AI tools in education.,An AI education policy should state which activities it addresses.,neutral,test,ai_education-family-08,assistant_authored_synthetic; requires independent human review
gd-ai_education-neutral-09,ai_education,Students should be allowed to use AI tools in education.,The group has not yet defined whether teacher-operated AI is included in the proposal.,neutral,test,ai_education-family-09,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-00,digital_payments,Digital payments are better than cash for everyday transactions.,Digital payments let people complete purchases without carrying exact change.,support,train,digital_payments-family-00,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-01,digital_payments,Digital payments are better than cash for everyday transactions.,Electronic transaction records make it easier to track everyday spending.,support,train,digital_payments-family-01,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-02,digital_payments,Digital payments are better than cash for everyday transactions.,Sending money digitally avoids travel when the recipient lives far away.,support,train,digital_payments-family-02,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-03,digital_payments,Digital payments are better than cash for everyday transactions.,Shopkeepers can receive payments quickly even when customers have no cash in their wallets.,support,train,digital_payments-family-03,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-04,digital_payments,Digital payments are better than cash for everyday transactions.,Digital transactions reduce the need to handle and store large amounts of physical currency.,support,train,digital_payments-family-04,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-05,digital_payments,Digital payments are better than cash for everyday transactions.,Automatic payment confirmations help buyers and sellers check that a transaction was completed.,support,train,digital_payments-family-05,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-06,digital_payments,Digital payments are better than cash for everyday transactions.,Recurring bills are easier to manage when payments can be scheduled electronically.,support,dev,digital_payments-family-06,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-07,digital_payments,Digital payments are better than cash for everyday transactions.,Small purchases become more convenient when customers can pay using a phone they already carry.,support,dev,digital_payments-family-07,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-08,digital_payments,Digital payments are better than cash for everyday transactions.,An electronic payment history helps people organize their household budgets.,support,test,digital_payments-family-08,assistant_authored_synthetic; requires independent human review
gd-digital_payments-support-09,digital_payments,Digital payments are better than cash for everyday transactions.,Contactless payment can shorten queues where counting cash slows down every transaction.,support,test,digital_payments-family-09,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-00,digital_payments,Digital payments are better than cash for everyday transactions.,Cash still works when internet services or payment servers are unavailable.,oppose,train,digital_payments-family-00,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-01,digital_payments,Digital payments are better than cash for everyday transactions.,People without smartphones or bank access can be excluded by a digital-only payment system.,oppose,train,digital_payments-family-01,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-02,digital_payments,Digital payments are better than cash for everyday transactions.,Digital fraud can move money rapidly before a victim understands what happened.,oppose,train,digital_payments-family-02,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-03,digital_payments,Digital payments are better than cash for everyday transactions.,Paying in cash does not require sharing a transaction history with multiple service providers.,oppose,train,digital_payments-family-03,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-04,digital_payments,Digital payments are better than cash for everyday transactions.,An empty phone battery can prevent a digital payment even when a customer has enough money.,oppose,train,digital_payments-family-04,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-05,digital_payments,Digital payments are better than cash for everyday transactions.,Some users find cash easier to budget because they can physically see what remains.,oppose,train,digital_payments-family-05,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-06,digital_payments,Digital payments are better than cash for everyday transactions.,Network failures can stop a shop from accepting digital payments during its busiest hours.,oppose,dev,digital_payments-family-06,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-07,digital_payments,Digital payments are better than cash for everyday transactions.,Customers may struggle to recover funds after sending a payment to the wrong digital address.,oppose,dev,digital_payments-family-07,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-08,digital_payments,Digital payments are better than cash for everyday transactions.,Older people unfamiliar with payment apps may need assistance for transactions they could complete independently with cash.,oppose,test,digital_payments-family-08,assistant_authored_synthetic; requires independent human review
gd-digital_payments-oppose-09,digital_payments,Digital payments are better than cash for everyday transactions.,Digital transaction fees can reduce the earnings of merchants operating on very small margins.,oppose,test,digital_payments-family-09,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-00,digital_payments,Digital payments are better than cash for everyday transactions.,"Digital payments include bank transfers, cards and mobile payment applications.",neutral,train,digital_payments-family-00,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-01,digital_payments,Digital payments are better than cash for everyday transactions.,We should define the size and frequency of the transactions being compared.,neutral,train,digital_payments-family-01,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-02,digital_payments,Digital payments are better than cash for everyday transactions.,Which payment method does the proposed digital system use?,neutral,train,digital_payments-family-02,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-03,digital_payments,Digital payments are better than cash for everyday transactions.,The discussion can consider customers and merchants separately.,neutral,train,digital_payments-family-03,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-04,digital_payments,Digital payments are better than cash for everyday transactions.,Cash consists of physical currency while digital payments transfer recorded balances.,neutral,train,digital_payments-family-04,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-05,digital_payments,Digital payments are better than cash for everyday transactions.,We need to specify whether transactions take place in cities or remote areas.,neutral,train,digital_payments-family-05,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-06,digital_payments,Digital payments are better than cash for everyday transactions.,Transaction speed and settlement time are different measures.,neutral,dev,digital_payments-family-06,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-07,digital_payments,Digital payments are better than cash for everyday transactions.,Some businesses accept several payment methods at the same counter.,neutral,dev,digital_payments-family-07,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-08,digital_payments,Digital payments are better than cash for everyday transactions.,Let us clarify whether this discussion includes online shopping as well as physical stores.,neutral,test,digital_payments-family-08,assistant_authored_synthetic; requires independent human review
gd-digital_payments-neutral-09,digital_payments,Digital payments are better than cash for everyday transactions.,A useful comparison would describe the devices each payment method requires.,neutral,test,digital_payments-family-09,assistant_authored_synthetic; requires independent human review
gd-startups-support-00,startups,Joining a startup is a better career choice than joining a large corporation.,Startup employees often gain experience across several functions instead of working on one narrow task.,support,train,startups-family-00,assistant_authored_synthetic; requires independent human review
gd-startups-support-01,startups,Joining a startup is a better career choice than joining a large corporation.,A small team can give a new employee meaningful responsibility early in their career.,support,train,startups-family-01,assistant_authored_synthetic; requires independent human review
gd-startups-support-02,startups,Joining a startup is a better career choice than joining a large corporation.,Working closely with founders helps employees understand how business decisions are made.,support,train,startups-family-02,assistant_authored_synthetic; requires independent human review
gd-startups-support-03,startups,Joining a startup is a better career choice than joining a large corporation.,Startups can move quickly enough for employees to see their ideas become real products.,support,train,startups-family-03,assistant_authored_synthetic; requires independent human review
gd-startups-support-04,startups,Joining a startup is a better career choice than joining a large corporation.,Early employees may have more freedom to shape their role and develop new skills.,support,train,startups-family-04,assistant_authored_synthetic; requires independent human review
gd-startups-support-05,startups,Joining a startup is a better career choice than joining a large corporation.,Building processes from scratch can provide valuable experience for someone who wants to become an entrepreneur.,support,train,startups-family-05,assistant_authored_synthetic; requires independent human review
gd-startups-support-06,startups,Joining a startup is a better career choice than joining a large corporation.,Fewer management layers can make it easier for junior staff to contribute directly to important decisions.,support,dev,startups-family-06,assistant_authored_synthetic; requires independent human review
gd-startups-support-07,startups,Joining a startup is a better career choice than joining a large corporation.,A startup role can offer broad ownership of a project from design through delivery.,support,dev,startups-family-07,assistant_authored_synthetic; requires independent human review
gd-startups-support-08,startups,Joining a startup is a better career choice than joining a large corporation.,Employees who enjoy experimentation may learn faster in a small company trying new approaches.,support,test,startups-family-08,assistant_authored_synthetic; requires independent human review
gd-startups-support-09,startups,Joining a startup is a better career choice than joining a large corporation.,Close contact with customers can help startup employees understand the impact of their work.,support,test,startups-family-09,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-00,startups,Joining a startup is a better career choice than joining a large corporation.,Large corporations usually offer more predictable salaries than startups with uncertain funding.,oppose,train,startups-family-00,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-01,startups,Joining a startup is a better career choice than joining a large corporation.,A new graduate may benefit from structured training that a small startup cannot provide.,oppose,train,startups-family-01,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-02,startups,Joining a startup is a better career choice than joining a large corporation.,Startup employees can face heavy workloads when a small team has too few resources.,oppose,train,startups-family-02,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-03,startups,Joining a startup is a better career choice than joining a large corporation.,An established company often offers clearer promotion paths and specialist mentors.,oppose,train,startups-family-03,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-04,startups,Joining a startup is a better career choice than joining a large corporation.,The risk of a startup closing can make it a poor choice for someone who needs stable income.,oppose,train,startups-family-04,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-05,startups,Joining a startup is a better career choice than joining a large corporation.,Large organizations may provide stronger benefits and more consistent support for employees.,oppose,train,startups-family-05,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-06,startups,Joining a startup is a better career choice than joining a large corporation.,A startup's changing priorities can leave employees with little clarity about their long-term role.,oppose,dev,startups-family-06,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-07,startups,Joining a startup is a better career choice than joining a large corporation.,Corporate experience can provide access to mature systems and projects at a scale a startup cannot match.,oppose,dev,startups-family-07,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-08,startups,Joining a startup is a better career choice than joining a large corporation.,New companies may lack the time to train inexperienced staff before expecting them to deliver results.,oppose,test,startups-family-08,assistant_authored_synthetic; requires independent human review
gd-startups-oppose-09,startups,Joining a startup is a better career choice than joining a large corporation.,An employee who values predictable working hours may find an established corporation more suitable.,oppose,test,startups-family-09,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-00,startups,Joining a startup is a better career choice than joining a large corporation.,We should define the candidate's career stage and professional goals.,neutral,train,startups-family-00,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-01,startups,Joining a startup is a better career choice than joining a large corporation.,"Startups vary widely in size, funding and age.",neutral,train,startups-family-01,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-02,startups,Joining a startup is a better career choice than joining a large corporation.,The comparison should specify whether the jobs involve similar responsibilities.,neutral,train,startups-family-02,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-03,startups,Joining a startup is a better career choice than joining a large corporation.,Which industry does the candidate want to work in?,neutral,train,startups-family-03,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-04,startups,Joining a startup is a better career choice than joining a large corporation.,"A corporation and a startup can both hire engineers, designers and sales staff.",neutral,train,startups-family-04,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-05,startups,Joining a startup is a better career choice than joining a large corporation.,The word startup does not describe one fixed salary or work schedule.,neutral,train,startups-family-05,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-06,startups,Joining a startup is a better career choice than joining a large corporation.,Let us clarify whether the candidate has financial responsibilities that affect the decision.,neutral,dev,startups-family-06,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-07,startups,Joining a startup is a better career choice than joining a large corporation.,We could compare two actual job descriptions instead of broad company categories.,neutral,dev,startups-family-07,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-08,startups,Joining a startup is a better career choice than joining a large corporation.,The discussion concerns career choices rather than investment in company shares.,neutral,test,startups-family-08,assistant_authored_synthetic; requires independent human review
gd-startups-neutral-09,startups,Joining a startup is a better career choice than joining a large corporation.,What time horizon should we use when comparing the two career paths?,neutral,test,startups-family-09,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-00,ev_india,India is ready for widespread adoption of electric vehicles.,Electric scooters already suit many predictable urban trips in India.,support,train,ev_india-family-00,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-01,ev_india,India is ready for widespread adoption of electric vehicles.,Growing domestic manufacturing can support a wider transition to electric vehicles.,support,train,ev_india-family-01,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-02,ev_india,India is ready for widespread adoption of electric vehicles.,Many drivers with dedicated parking can charge their electric vehicles at home overnight.,support,train,ev_india-family-02,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-03,ev_india,India is ready for widespread adoption of electric vehicles.,Indian delivery fleets can use fixed routes and depot charging to adopt electric vehicles at scale.,support,train,ev_india-family-03,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-04,ev_india,India is ready for widespread adoption of electric vehicles.,Increasing charging options in major cities make electric transport practical for more drivers.,support,train,ev_india-family-04,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-05,ev_india,India is ready for widespread adoption of electric vehicles.,Local engineering and servicing skills can expand alongside the Indian electric vehicle market.,support,train,ev_india-family-05,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-06,ev_india,India is ready for widespread adoption of electric vehicles.,Short daily commutes allow many households to meet their transport needs with existing battery ranges.,support,dev,ev_india-family-06,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-07,ev_india,India is ready for widespread adoption of electric vehicles.,"Fleet operators can plan charging schedules centrally, making large-scale electrification workable.",support,dev,ev_india-family-07,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-08,ev_india,India is ready for widespread adoption of electric vehicles.,Apartment charging programs can extend home charging access to more urban residents.,support,test,ev_india-family-08,assistant_authored_synthetic; requires independent human review
gd-ev_india-support-09,ev_india,India is ready for widespread adoption of electric vehicles.,India can begin widespread adoption through two-wheelers and buses without waiting for every vehicle category to change.,support,test,ev_india-family-09,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-00,ev_india,India is ready for widespread adoption of electric vehicles.,Many rural routes still lack dependable public charging facilities.,oppose,train,ev_india-family-00,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-01,ev_india,India is ready for widespread adoption of electric vehicles.,Apartment residents without assigned parking may have no practical place to charge an electric vehicle.,oppose,train,ev_india-family-01,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-02,ev_india,India is ready for widespread adoption of electric vehicles.,High purchase prices can keep electric vehicles out of reach for many Indian households.,oppose,train,ev_india-family-02,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-03,ev_india,India is ready for widespread adoption of electric vehicles.,Long-distance drivers need more reliable charging coverage before electric vehicles can replace their current vehicles.,oppose,train,ev_india-family-03,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-04,ev_india,India is ready for widespread adoption of electric vehicles.,Uneven electricity supply makes overnight charging unreliable in some areas.,oppose,train,ev_india-family-04,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-05,ev_india,India is ready for widespread adoption of electric vehicles.,India needs more trained repair technicians outside major cities before adoption can be widespread.,oppose,train,ev_india-family-05,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-06,ev_india,India is ready for widespread adoption of electric vehicles.,Battery replacement costs create uncertainty for buyers with limited savings.,oppose,dev,ev_india-family-06,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-07,ev_india,India is ready for widespread adoption of electric vehicles.,Public chargers that are broken or difficult to access can make journeys unpredictable.,oppose,dev,ev_india-family-07,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-08,ev_india,India is ready for widespread adoption of electric vehicles.,Heavy commercial vehicles face charging and range requirements that existing infrastructure may not support.,oppose,test,ev_india-family-08,assistant_authored_synthetic; requires independent human review
gd-ev_india-oppose-09,ev_india,India is ready for widespread adoption of electric vehicles.,Charging standards and payment systems need better coordination before drivers can rely on a nationwide network.,oppose,test,ev_india-family-09,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-00,ev_india,India is ready for widespread adoption of electric vehicles.,"Electric vehicles include two-wheelers, cars, buses and commercial vehicles.",neutral,train,ev_india-family-00,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-01,ev_india,India is ready for widespread adoption of electric vehicles.,We should define whether widespread adoption means new sales or the entire vehicle fleet.,neutral,train,ev_india-family-01,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-02,ev_india,India is ready for widespread adoption of electric vehicles.,The discussion needs a time period for assessing India's readiness.,neutral,train,ev_india-family-02,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-03,ev_india,India is ready for widespread adoption of electric vehicles.,Charging at home and charging on highways involve different infrastructure.,neutral,train,ev_india-family-03,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-04,ev_india,India is ready for widespread adoption of electric vehicles.,Which states and types of journeys are included in this comparison?,neutral,train,ev_india-family-04,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-05,ev_india,India is ready for widespread adoption of electric vehicles.,Battery size influences how often an electric vehicle needs charging.,neutral,train,ev_india-family-05,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-06,ev_india,India is ready for widespread adoption of electric vehicles.,The group could assess urban transport and rural transport separately.,neutral,dev,ev_india-family-06,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-07,ev_india,India is ready for widespread adoption of electric vehicles.,We should distinguish battery manufacturing from the installation of charging points.,neutral,dev,ev_india-family-07,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-08,ev_india,India is ready for widespread adoption of electric vehicles.,"The term readiness could refer to technology, affordability or infrastructure.",neutral,test,ev_india-family-08,assistant_authored_synthetic; requires independent human review
gd-ev_india-neutral-09,ev_india,India is ready for widespread adoption of electric vehicles.,Let us agree on how we would measure the availability of a usable charger.,neutral,test,ev_india-family-09,assistant_authored_synthetic; requires independent human review
gd-environment-support-00,environment,Environmental protection should take priority over short-term economic growth.,Protecting clean water should come before projects that offer temporary profits while damaging local health.,support,train,environment-family-00,assistant_authored_synthetic; requires independent human review
gd-environment-support-01,environment,Environmental protection should take priority over short-term economic growth.,Destroying forests for short-term growth can undermine livelihoods that depend on them for decades.,support,train,environment-family-01,assistant_authored_synthetic; requires independent human review
gd-environment-support-02,environment,Environmental protection should take priority over short-term economic growth.,Pollution cleanup can cost more than the immediate economic gains from ignoring environmental damage.,support,train,environment-family-02,assistant_authored_synthetic; requires independent human review
gd-environment-support-03,environment,Environmental protection should take priority over short-term economic growth.,Healthy ecosystems provide services that future businesses and communities need to survive.,support,train,environment-family-03,assistant_authored_synthetic; requires independent human review
gd-environment-support-04,environment,Environmental protection should take priority over short-term economic growth.,Short-lived economic benefits do not justify environmental harm that cannot be reversed.,support,train,environment-family-04,assistant_authored_synthetic; requires independent human review
gd-environment-support-05,environment,Environmental protection should take priority over short-term economic growth.,Protecting natural resources preserves economic opportunities for future generations.,support,train,environment-family-05,assistant_authored_synthetic; requires independent human review
gd-environment-support-06,environment,Environmental protection should take priority over short-term economic growth.,Environmental safeguards can prevent health costs that would otherwise be passed to families and public services.,support,dev,environment-family-06,assistant_authored_synthetic; requires independent human review
gd-environment-support-07,environment,Environmental protection should take priority over short-term economic growth.,Wetlands can protect communities from floods more sustainably than rebuilding after repeated damage.,support,dev,environment-family-07,assistant_authored_synthetic; requires independent human review
gd-environment-support-08,environment,Environmental protection should take priority over short-term economic growth.,Communities should not sacrifice long-term access to safe air and water for a brief increase in output.,support,test,environment-family-08,assistant_authored_synthetic; requires independent human review
gd-environment-support-09,environment,Environmental protection should take priority over short-term economic growth.,Avoiding irreversible habitat loss deserves priority even when a damaging project promises quick revenue.,support,test,environment-family-09,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-00,environment,Environmental protection should take priority over short-term economic growth.,Communities facing severe unemployment may need immediate economic growth before they can afford stronger environmental programs.,oppose,train,environment-family-00,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-01,environment,Environmental protection should take priority over short-term economic growth.,Blocking every resource-intensive project can remove essential income from families with few alternatives.,oppose,train,environment-family-01,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-02,environment,Environmental protection should take priority over short-term economic growth.,Economic growth can fund cleaner technology that poorer communities cannot currently purchase.,oppose,train,environment-family-02,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-03,environment,Environmental protection should take priority over short-term economic growth.,A rigid environmental priority can delay infrastructure needed to improve basic living conditions.,oppose,train,environment-family-03,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-04,environment,Environmental protection should take priority over short-term economic growth.,Workers should not lose their livelihoods because environmental rules ignore the cost of a transition.,oppose,train,environment-family-04,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-05,environment,Environmental protection should take priority over short-term economic growth.,Governments need tax revenue from growing businesses to finance public health and environmental restoration.,oppose,train,environment-family-05,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-06,environment,Environmental protection should take priority over short-term economic growth.,Poor regions may reasonably prioritize reliable electricity and industry while gradually reducing their environmental impact.,oppose,dev,environment-family-06,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-07,environment,Environmental protection should take priority over short-term economic growth.,Immediate development can provide education and services that help communities manage future environmental risks.,oppose,dev,environment-family-07,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-08,environment,Environmental protection should take priority over short-term economic growth.,An abrupt halt to economic activity can create hardship greater than the short-term environmental benefit.,oppose,test,environment-family-08,assistant_authored_synthetic; requires independent human review
gd-environment-oppose-09,environment,Environmental protection should take priority over short-term economic growth.,Environmental goals should not automatically overrule projects that meet urgent needs for housing and employment.,oppose,test,environment-family-09,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-00,environment,Environmental protection should take priority over short-term economic growth.,We need to define which environmental impacts and which economic indicators are being compared.,neutral,train,environment-family-00,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-01,environment,Environmental protection should take priority over short-term economic growth.,The discussion should specify what short-term means in years.,neutral,train,environment-family-01,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-02,environment,Environmental protection should take priority over short-term economic growth.,"Different projects affect air, water and land in different ways.",neutral,train,environment-family-02,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-03,environment,Environmental protection should take priority over short-term economic growth.,Who will measure the costs and benefits of the proposed development?,neutral,train,environment-family-03,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-04,environment,Environmental protection should take priority over short-term economic growth.,"Environmental protection can include conservation, pollution control and restoration.",neutral,train,environment-family-04,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-05,environment,Environmental protection should take priority over short-term economic growth.,"Economic growth is often measured using output, income or employment.",neutral,train,environment-family-05,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-06,environment,Environmental protection should take priority over short-term economic growth.,We should identify which communities receive the benefits and which face the costs.,neutral,dev,environment-family-06,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-07,environment,Environmental protection should take priority over short-term economic growth.,The group could compare a particular industrial project with a particular conservation proposal.,neutral,dev,environment-family-07,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-08,environment,Environmental protection should take priority over short-term economic growth.,Let us distinguish irreversible damage from effects that can be repaired.,neutral,test,environment-family-08,assistant_authored_synthetic; requires independent human review
gd-environment-neutral-09,environment,Environmental protection should take priority over short-term economic growth.,The proposal needs a clearly stated location and timescale before we assess its consequences.,neutral,test,environment-family-09,assistant_authored_synthetic; requires independent human review
'''
(PROJECT / 'data' / 'gd_examples.csv').write_text(GD_CSV, encoding='utf-8')
print('Saved the 300 labelled GD examples.')


In [ ]:
EXTRA_FILES = {'requirements.txt': 'numpy==2.5.4\nscipy==1.16.3\nscikit-learn==1.7.2\njoblib==1.6.0\nmatplotlib==3.10.7\n', 'START_HERE.txt': 'GD ARENA — PYTHON ML MODEL, PHASE 1\n==================================\n\nThis package contains the actual training pipeline and saved model for the\n10-topic Group Discussion project. This phase is the model, not the voice UI.\n\nEASIEST WAY: GOOGLE COLAB\n1. Open https://colab.research.google.com/\n2. File > Upload notebook > select GD_Arena_Colab.ipynb.\n3. Use a normal Python 3 CPU runtime. GPU is not required.\n4. Runtime > Run all. Internet is needed for packages and the public dataset.\n5. Edit topic_id and your English argument in the prediction cell.\n6. The final cell saves a ZIP of your trained model, code, metrics and figures.\n   In Colab it also provides a download link.\n\nThe notebook is self-contained: it writes the Python module, the 10 topic\ndefinitions and the 300 example rows. You do not need to upload separate .py\nor CSV files. If Colab asks for a runtime restart after installation, restart\nand run all again. The notebook was checked locally; no claim is made that it\nwas run inside your Google account.\n\nLOCAL PYTHON (tested with Python 3.12)\n  python -m venv .venv\n  .venv\\Scripts\\activate\n  python -m pip install -r requirements.txt\n  python gd_model.py topics\n  python gd_model.py predict --topic remote_work --text "Working from home saves commuting time and allows focused work."\n\nTo retrain from scratch:\n  python gd_model.py train\n\nThe first training run downloads the authors\' VAST files and verifies their\nSHA-256 checksums. Do not edit the downloaded VAST files. Put additional GD\nexamples in data/gd_examples.csv with explicit labels and split/group fields.\n\nPython API for the future UI:\n  import joblib\n  from gd_model import predict\n  model = joblib.load("artifacts/stance_model.joblib")\n  result = predict(model, "Your English argument here", topic_id="remote_work")\n  print(result)\n\nOnly load joblib/pickle model files from a trusted source. Keep gd_model.py\nbeside your calling code so Python can import the saved feature transformer.\n\nWHAT IS ACTUALLY MACHINE LEARNING?\n- TF-IDF features learned from the training text and topic targets.\n- Word n-grams, character n-grams and topic-word interaction features.\n- Logistic Regression and Linear SVM trained on labelled examples.\n- A majority-label baseline for comparison.\n- Model selection using validation macro F1, followed by held-out testing.\n- No external AI API and no paid service are required for training/inference.\n\nWHAT THE LABELS MEAN\nSupport / oppose are relative to the explicit target proposition in topics.json.\nThe display title may be a question; the target states exactly which side\n"support" refers to. Neutral means no clear stance towards that proposition.\nNeutral is not a low-confidence class. Stance is not argument quality.\nReturned decision scores are uncalibrated and must not be shown as percentages.\n\nHONEST PROJECT SCOPE\nThis is an experimental classical ML baseline. It does not generate discussion\nresponses, recognize speech, assess interview suitability, or understand\nHindi/Hinglish. Those are separate future components. Neither a fluent reply\nnor a supportive stance proves that an argument is correct or well reasoned.\n\nThe 300 GD examples were authored by the assistant, with labels assigned by\nthe assistant. They are not collected or independently annotated human data.\nTheir small test set is a development check, not real-world validation.\nCollect and independently label real GD responses before claiming practical\naccuracy. Keep new final evaluation data separate from training and tuning.\n\nFILES\n  gd_model.py                  Complete training / inference source\n  topics.json                  The 10 agreed topics and target propositions\n  data/gd_examples.csv         300 explicitly authored labelled examples\n  requirements.txt             Pinned ML runtime dependencies\n  artifacts/stance_model.joblib Saved fitted feature transformer + classifier\n  artifacts/metrics.json        Full measured results and data-cleaning audit\n  artifacts/*_predictions.csv    Auditable test predictions by row ID\n  artifacts/*.png               Evaluation graphs\n  RESULTS.txt                   Readable results generated after training\n  DATA_SOURCES.txt              Dataset provenance, citation and reuse notes\n  tests/test_model.py           Leakage, persistence and input checks\n\nThe UI can later call predict() on a reviewed English transcript. Speech\nrecognition, translation and virtual-participant replies must be evaluated\nseparately; this model\'s test results do not measure those components.\n', 'DATA_SOURCES.txt': "DATA PROVENANCE\n===============\n\n1. VAST: VAried Stance Topics\nAuthors: Emily Allaway and Kathleen McKeown (2020).\nPaper: Zero-Shot Stance Detection: A Dataset and Model using Generalized Topic\nRepresentations. Proceedings of EMNLP 2020, pages 8913-8931.\nhttps://aclanthology.org/2020.emnlp-main.717/\nOriginal author repository:\nhttps://github.com/emilyallaway/zero-shot-stance\nPinned revision: e7c4775182b184730f350995f8260579c9e066fe\nOriginal split files: data/VAST/vast_train.csv, vast_dev.csv, vast_test.csv.\n\nMap original label 0 to oppose, 1 to support, 2 to neutral.\nUse the original post as argument text and the JSON topic token list as target.\nImportant: new_topic can retain a stale target in neutral examples; it must\nnot be used as the stance target. This matches the authors' target selection:\nhttps://github.com/emilyallaway/zero-shot-stance/blob/master/src/modeling/datasets.py\nThe upstream\nprocessed text is not used because its cleaning removes useful words such as\nnegations. Official splits are retained, then exact normalized text leakage,\nduplicate text-target pairs and conflicting pair labels are audited/removed.\nThe metrics therefore describe our cleaned splits, not an exact reproduction\nof the paper's benchmark. Topics with different spellings may still overlap\nsemantically even when exact strings do not.\n\nThe checked author repository did not provide an explicit standalone dataset\nlicense. Public availability is not a blanket redistribution license. Raw\nVAST comments are not bundled in the deliverable. The notebook downloads the\nauthors' release directly for this educational experiment. Consult the\nauthors/source terms before wider redistribution or commercial reuse. This\npackage does not assign a new license to third-party data or derived weights.\n\n2. GD Arena example set\nFile: data/gd_examples.csv\n300 English rows: 10 topics x 3 labels x 10 examples.\nProvenance: original assistant-authored synthetic educational examples.\nLabels: assistant-assigned, not independently verified.\nSplit: 180 training, 60 validation, 60 test examples, fixed before training.\nThe numbered topic-family group is kept in one split; no automatic paraphrase\nor duplicated-template expansion is used to inflate the dataset size.\nThis is a starter domain-adaptation set. Some arguments are closely related;\nits test scores must not be presented as independent human benchmark results.\n\nThe content is hypothetical debate material, not verified factual evidence.\nTopic IDs are not predictive features; the full target text is used instead.\nThe source column and split labels are not passed to the classifier.\n\nANNOTATING YOUR OWN DATA\nWrite a clear target proposition first. Label a response support if it favors\nthat proposition, oppose if it rejects it, and neutral if it expresses no\nclear stance. Mixed or ambiguous statements need a written annotation policy.\nAsk two people to label without seeing model predictions; adjudicate conflicts.\nKeep paraphrases, the same speaker/session and repeated source text in the\nsame split. Reserve a fresh real-user test set before tuning your model.\n\nReferences for implementation:\nhttps://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction\nhttps://scikit-learn.org/stable/common_pitfalls.html#data-leakage\nhttps://research.google.com/colaboratory/faq.html\n", 'tests/test_model.py': "import json\nimport csv\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom gd_model import LABELS, StanceFeatures, key, predict, read_gd, read_vast\n\n\nclass ModelChecks(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.bundle = joblib.load(ROOT / 'artifacts' / 'stance_model.joblib')\n        cls.gd = read_gd(ROOT / 'data' / 'gd_examples.csv')\n\n    def test_ten_topics_three_classes(self):\n        self.assertEqual(len(self.bundle['topics']), 10)\n        self.assertEqual(set(self.bundle['classifier'].classes_), set(LABELS))\n        for split, expected in [('train', 180), ('dev', 60), ('test', 60)]:\n            self.assertEqual(len(self.gd[split]), expected)\n\n    def test_vast_neutral_uses_actual_target_not_stale_new_topic(self):\n        with tempfile.TemporaryDirectory() as directory:\n            folder = Path(directory)\n            for split in ['train', 'dev', 'test']:\n                with (folder / f'vast_{split}.csv').open('w', newline='', encoding='utf-8') as handle:\n                    writer = csv.DictWriter(handle, fieldnames=['post', 'topic', 'new_topic', 'label', 'seen?'])\n                    writer.writeheader()\n                    if split == 'train':\n                        for target, label in [(['remote', 'work'], '1'), (['school', 'uniforms'], '2')]:\n                            writer.writerow({'post': 'Remote work saves commuting time.',\n                                             'topic': json.dumps(target), 'new_topic': 'remote work',\n                                             'label': label, 'seen?': '0'})\n            rows, audit = read_vast(folder)\n            self.assertEqual(len(rows['train']), 2)\n            self.assertEqual(rows['train'][1]['topic'], 'school uniforms')\n            self.assertEqual(rows['train'][1]['label'], 'neutral')\n            self.assertNotIn('conflicting_pair', audit['train']['removed'])\n\n    def test_document_and_family_isolation(self):\n        for left, right in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:\n            for field in ['text', 'group']:\n                a = {key(r[field]) for r in self.gd[left]}\n                b = {key(r[field]) for r in self.gd[right]}\n                self.assertFalse(a & b)\n\n    def test_target_changes_features(self):\n        rows = [{'text': 'Home working saves travel time.', 'topic': 'working from home'},\n                {'text': 'Home working saves travel time.', 'topic': 'classroom education'}]\n        x = self.bundle['features'].transform(rows)\n        self.assertGreater((x[0] - x[1]).nnz, 0)\n\n    def test_heldout_vocabulary_is_not_fitted(self):\n        rows = [{'topic': 'remote work', 'text': 'Remote workers save commuting time.'},\n                {'topic': 'remote work', 'text': 'Workers need commuting time for office work.'},\n                {'topic': 'remote work', 'text': 'Remote work helps workers save time.'}]\n        features = StanceFeatures().fit(rows)\n        features.transform([{'topic': 'remote work', 'text': 'quasarholdouttoken appears only in heldout text.'}])\n        self.assertNotIn('quasarholdouttoken', features.arguments_.vocabulary_)\n\n    def test_saved_model_roundtrip(self):\n        text = 'Working from home saves commuting time and allows focused work.'\n        before = predict(self.bundle, text, topic_id='remote_work')\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / 'copy.joblib'\n            joblib.dump(self.bundle, path)\n            after = predict(joblib.load(path), text, topic_id='remote_work')\n        self.assertEqual(before, after)\n\n    def test_invalid_inputs(self):\n        for text in ['', 'hello', None, 'a' * 5001]:\n            with self.assertRaises(ValueError):\n                predict(self.bundle, text, topic_id='remote_work')\n        with self.assertRaises(ValueError):\n            predict(self.bundle, 'This is a valid argument.', topic_id='missing')\n        with self.assertRaises(ValueError):\n            predict(self.bundle, 'This is a valid argument.')\n\n    def test_prediction_schema_all_topics(self):\n        for topic in self.bundle['topics']:\n            output = predict(self.bundle, 'I would like to understand the issue before taking a position.', topic_id=topic['id'])\n            self.assertIn(output['stance'], LABELS)\n            self.assertEqual(set(output['decision_scores']), set(LABELS))\n            self.assertTrue(all(np.isfinite(v) for v in output['decision_scores'].values()))\n\n    def test_metrics_match_saved_model(self):\n        report = json.loads((ROOT / 'artifacts' / 'metrics.json').read_text())\n        self.assertEqual(report, self.bundle['report'])\n        expected = max(['Logistic Regression', 'Linear SVM'],\n                       key=lambda n: report['validation'][n]['selection_score'])\n        self.assertEqual(expected, report['selected_model'])\n        for dataset in ['vast', 'gd']:\n            cm = np.array(report['test'][dataset]['confusion_matrix'])\n            self.assertEqual(int(cm.sum()), report['test'][dataset]['n'])\n            self.assertAlmostEqual(float(np.trace(cm) / cm.sum()), report['test'][dataset]['accuracy'])\n\n\nif __name__ == '__main__':\n    unittest.main()\n"}
for name, content in EXTRA_FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(content, encoding='utf-8')


## 5. Download and audit the public dataset
VAST was introduced by **Emily Allaway and Kathleen McKeown (2020)**:
[paper](https://aclanthology.org/2020.emnlp-main.717/) ·
[original source](https://github.com/emilyallaway/zero-shot-stance).

The download is pinned to a Git revision and verified by SHA-256. Original labels are mapped as
0 → oppose, 1 → support, 2 → neutral. We retain original posts so words such as “not” are preserved.
The actual target comes from the JSON `topic` tokens, as in the authors' loader. The `new_topic`
column can be stale for neutral examples and is deliberately not used.
Only text and target are predictive inputs; authors, IDs, split fields and labels are excluded.

We retain the authors' split boundaries, remove conflicting duplicate pairs and remove documents
already present in an earlier split. Reported metrics therefore refer to our cleaned splits.
They are not directly comparable to the paper's exact benchmark. The source repository has no
explicit standalone dataset license; raw third-party comments are not included in the downloadable
project ZIP. Check upstream terms before wider redistribution or commercial reuse.


In [ ]:
import sys
sys.path.insert(0, str(PROJECT))
from gd_model import download_vast, read_vast, read_gd, train, predict

# Defaults to this notebook's folder. The optional environment override supports local verification.
DATA_DIR = Path(os.environ.get('GD_VAST_DATA', str(PROJECT / 'data' / 'vast')))
download_vast(DATA_DIR)
vast, audit = read_vast(DATA_DIR)
gd = read_gd(PROJECT / 'data' / 'gd_examples.csv')
for split in ['train', 'dev', 'test']:
    print(split, '| VAST:', len(vast[split]), '| authored GD:', len(gd[split]),
          '| VAST labels:', audit[split]['class_counts'])
print('Cleaning audit:', json.dumps(audit, indent=2))


## 6. Train and compare models
Feature blocks: word TF-IDF, character TF-IDF, target TF-IDF and target–argument lexical
interactions. All vocabularies and IDF weights are fitted **only on training rows**.

Compare a majority-label baseline, **Logistic Regression** and **Linear SVM**. The small GD
training set receives a fixed sample weight of 8 versus 1 for VAST. Select the learned model
using the mean of the two validation macro-F1 values. These choices are fixed before testing.
The selected model stays trained on the training split so the saved model exactly matches
the reported evaluation. No test-set tuning or post-test refitting is done.

Macro F1 gives each stance class equal weight. Accuracy alone can hide poor minority-class results.
The authored GD score is only a limited development check; VAST is reported separately.


In [ ]:
model, report = train(DATA_DIR, PROJECT / 'data' / 'gd_examples.csv', PROJECT / 'artifacts')
print('Selected model:', report['selected_model'])
for name, result in report['validation'].items():
    print(f"{name:22s} | VAST dev F1={result['vast']['macro_f1']:.3f} | "
          f"GD dev F1={result['gd']['macro_f1']:.3f} | selection={result['selection_score']:.3f}")
(PROJECT / 'RESULTS.txt').write_text(json.dumps(report, indent=2), encoding='utf-8')


## 7. Inspect held-out results
Test labels were not used to choose the model. Read per-class recall as well as the average.
Authored GD results must not be described as accuracy on real interview participants.
The exact unseen-target-string subset still may contain semantically related topics.


In [ ]:
from IPython.display import display, Image

for dataset in ['vast', 'gd', 'vast_unseen_target_strings']:
    if dataset not in report['test']:
        continue
    result = report['test'][dataset]
    print(f"{dataset}: n={result['n']}, accuracy={result['accuracy']:.3f}, macro F1={result['macro_f1']:.3f}")
    for label in ['oppose', 'support', 'neutral']:
        row = result['per_class'][label]
        print(f"  {label:8s} precision={row['precision']:.3f} recall={row['recall']:.3f} F1={row['f1-score']:.3f}")
display(Image(filename=str(PROJECT / 'artifacts' / 'model_comparison.png')))
display(Image(filename=str(PROJECT / 'artifacts' / 'confusion_matrices.png')))


## 8. Try your own argument
Change `topic_id` and `argument`. Use one of the ten IDs printed above.
The output is an experimental stance prediction. Raw decision scores are **not confidence
percentages** and do not measure whether your argument is true, persuasive or interview-ready.
English is supported; Hindi/Hinglish and speech are future app components.


In [ ]:
topic_id = 'remote_work'
argument = 'Working from home saves commuting time and lets employees concentrate without office interruptions.'
result = predict(model, argument, topic_id=topic_id)
print(json.dumps(result, indent=2))


## 9. Load the saved model in a fresh Python call
Keep `gd_model.py` importable next to the model. Load only trusted joblib files.


In [ ]:
import joblib
loaded_model = joblib.load(PROJECT / 'artifacts' / 'stance_model.joblib')
loaded_result = predict(loaded_model, argument, topic_id=topic_id)
assert loaded_result == result
print('Saved model reload check passed.')


## 10. Save your work
Download the ZIP after training. It contains your trained model, source, topic configuration,
authored examples and evaluation files. The original VAST comments are excluded.
Colab runtimes are temporary, so save the ZIP before closing your session.


In [ ]:
import zipfile

ZIP_PATH = Path.cwd() / 'GD_Arena_Trained_Project.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob('*'):
        if not path.is_file() or '__pycache__' in path.parts or 'vast' in path.relative_to(PROJECT).parts:
            continue
        archive.write(path, arcname=str(Path('gd_arena') / path.relative_to(PROJECT)))
print('Saved:', ZIP_PATH)
try:
    from google.colab import files
except ImportError:
    print('Outside Colab: open the ZIP from the current folder.')
else:
    files.download(str(ZIP_PATH))


## How to explain the project honestly
- “I trained and compared two classical ML classifiers using topic-conditioned TF-IDF features.”
- “The research dataset and the assistant-authored GD examples have separate evaluation results.”
- “The model detects stance; it does not score argument quality or generate dialogue.”
- “The next phase adds the UI and speech pipeline, with separate testing for each.”

Before deployment, collect independent real GD responses, review labels and evaluate them without
using that test set to tune the model. Negation, sarcasm and implicit stance are important failure cases.
